# Descriptive Statistics: Ghanaian Mobile Money Credit Risk Data

Systematic statistical summary of the synthetic MoMo dataset — numerical and categorical features, variability, distributions, and per-archetype breakdowns.

In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(project_root, 'src'))
from seqcredit_model.config import (
    DATA_DIR, TRANSACTIONS_DIR, USER_FEATURES_FILE, USER_LABELS_FILE,
    SYNTHETIC_PARAMS_FILE
)

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 100,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

ARCHETYPE_COLORS = {
    'non_borrower': '#95a5a6',
    'responsible_borrower': '#2ecc71',
    'occasional_borrower': '#3498db',
    'risky_borrower': '#f39c12',
    'defaulter': '#e74c3c',
}
ARCHETYPE_ORDER = [
    'non_borrower', 'responsible_borrower', 'occasional_borrower',
    'risky_borrower', 'defaulter'
]
ARCHETYPE_LABELS = {a: a.replace('_', ' ').title() for a in ARCHETYPE_ORDER}
RISK_COLORS = {-1: '#95a5a6', 0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}
RISK_LABELS = {-1: 'No Loans', 0: 'Good', 1: 'Late', 2: 'Default'}

print(f'Data directory: {DATA_DIR}')

In [ ]:
labels = pd.read_csv(str(USER_LABELS_FILE))
features = pd.read_csv(str(USER_FEATURES_FILE))
with open(str(SYNTHETIC_PARAMS_FILE)) as f:
    calibration = json.load(f)

users = features.merge(labels, on='user_id', how='inner')
users['total_transactions'] = users['total_transactions_x'] + users['total_transactions_y']

n_txn_files = len(list(TRANSACTIONS_DIR.glob('USER_*.csv')))
print(f'Users:             {len(users):,}')
print(f'Feature columns:   {features.shape[1] - 1}')
print(f'Label columns:     {labels.shape[1] - 1}')
print(f'Transaction files: {n_txn_files:,}')

In [ ]:
# Stratified sample of per-user transaction CSVs
np.random.seed(42)
SAMPLE_SIZE = 500

sample_ids = []
for archetype in ARCHETYPE_ORDER:
    arch_users = labels[labels['credit_archetype'] == archetype]['user_id'].values
    n_draw = max(10, int(SAMPLE_SIZE * len(arch_users) / len(labels)))
    drawn = np.random.choice(arch_users, size=min(n_draw, len(arch_users)), replace=False)
    sample_ids.extend(drawn)

txn_frames = []
for uid in sample_ids:
    fpath = TRANSACTIONS_DIR / f'{uid}.csv'
    if fpath.exists():
        df = pd.read_csv(str(fpath))
        df['user_id'] = uid
        txn_frames.append(df)

txns = pd.concat(txn_frames, ignore_index=True)
txns['TRANSACTION DATE'] = pd.to_datetime(txns['TRANSACTION DATE'])
txns = txns.merge(labels[['user_id', 'credit_archetype', 'credit_risk_label']], on='user_id')
txns['hour'] = txns['TRANSACTION DATE'].dt.hour
txns['day_of_week'] = txns['TRANSACTION DATE'].dt.day_name()

print(f'Sampled {len(set(sample_ids))} users -> {len(txns):,} transactions')
print(f'Date range: {txns["TRANSACTION DATE"].min().date()} to {txns["TRANSACTION DATE"].max().date()}')

---
## Section 1: Dataset Overview

In [ ]:
feature_cols = [c for c in features.columns if c != 'user_id']
num_cols = features[feature_cols].select_dtypes(include=np.number).columns.tolist()

print('=== USER FEATURES (user_features.csv) ===')
print(f'Shape: {features.shape[0]:,} rows x {features.shape[1]} columns')
print(f'Numeric feature columns: {len(num_cols)}')
print(f'Missing values: {features[num_cols].isnull().sum().sum()}')

print('\n=== USER LABELS (user_labels.csv) ===')
print(f'Shape: {labels.shape[0]:,} rows x {labels.shape[1]} columns')
print(f'Columns: {list(labels.columns)}')

print('\n=== SAMPLED TRANSACTION DATA ===')
print(f'Shape: {len(txns):,} rows x {txns.shape[1]} columns')
print(f'Columns: {list(txns.columns)}')

print('\n=== NUMERIC FEATURE COLUMNS ===')
for i, col in enumerate(num_cols, 1):
    print(f'  {i:2d}. {col}')

---
## Section 2: Numerical Feature Statistics

Full descriptive statistics for all numeric user-level features. CV (coefficient of variation = std/mean) measures relative variability — higher CV means more spread relative to the mean.

In [ ]:
stats = features[num_cols].describe().T
stats.insert(3, 'cv', (stats['std'] / stats['mean'].abs()).round(4))
stats.columns = ['Count', 'Mean', 'Std Dev', 'CV', 'Min', 'P25', 'Median', 'P75', 'Max']
stats = stats.round(4)

try:
    styled = (
        stats.style
        .format('{:.4f}')
        .background_gradient(subset=['CV'], cmap='YlOrRd')
        .background_gradient(subset=['Std Dev'], cmap='Blues')
    )
    display(styled)
except Exception:
    display(stats)

In [ ]:
# Top 15 features by coefficient of variation (most variable features)
cv_series = (features[num_cols].std() / features[num_cols].mean().abs()).sort_values(ascending=False)
top_cv = cv_series.head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(range(len(top_cv)), top_cv.values,
               color=plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(top_cv))), edgecolor='white')
ax.set_yticks(range(len(top_cv)))
ax.set_yticklabels([f.replace('_', ' ').title() for f in top_cv.index], fontsize=10)
ax.invert_yaxis()
for i, v in enumerate(top_cv.values):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)
ax.set_xlabel('Coefficient of Variation (std / |mean|)')
ax.set_title('Top 15 Features by Relative Variability (CV)')
plt.tight_layout()
plt.show()

print('\nBottom 5 (least variable):')
print(cv_series.tail(5).round(4).to_string())

### 2.1 Numerical Distributions by Archetype

Violin plots show the full distribution shape (median, IQR, tails) for key features across the five user archetypes.

In [ ]:
palette = [ARCHETYPE_COLORS[a] for a in ARCHETYPE_ORDER]
xlabels = [ARCHETYPE_LABELS[a] for a in ARCHETYPE_ORDER]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for (col, title), ax in zip([
    ('avg_balance', 'Avg Balance (GHS)'),
    ('balance_volatility', 'Balance Volatility (GHS)'),
    ('pct_low_balance_txns', 'Low-Balance Txn %'),
], axes):
    sns.violinplot(data=users, x='credit_archetype', y=col,
                   order=ARCHETYPE_ORDER, palette=palette, inner='box', ax=ax, cut=0)
    ax.set_xticklabels([a[:4].upper() for a in ARCHETYPE_ORDER], fontsize=9)
    ax.set_xlabel('')
    ax.set_title(title)
    for i, arch in enumerate(ARCHETYPE_ORDER):
        m = users[users['credit_archetype'] == arch][col].mean()
        s = users[users['credit_archetype'] == arch][col].std()
        ax.text(i, ax.get_ylim()[1] * 0.97, f'μ={m:.1f}\nσ={s:.1f}',
                ha='center', va='top', fontsize=8, color='black')
plt.suptitle('Balance Feature Distributions by Archetype', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for (col, title), ax in zip([
    ('total_transactions', 'Total Transactions'),
    ('avg_transaction_amount', 'Avg Txn Amount (GHS)'),
    ('cv_transaction_amount', 'CV of Txn Amounts'),
], axes):
    sns.violinplot(data=users, x='credit_archetype', y=col,
                   order=ARCHETYPE_ORDER, palette=palette, inner='box', ax=ax, cut=0)
    ax.set_xticklabels([a[:4].upper() for a in ARCHETYPE_ORDER], fontsize=9)
    ax.set_xlabel('')
    ax.set_title(title)
    for i, arch in enumerate(ARCHETYPE_ORDER):
        m = users[users['credit_archetype'] == arch][col].mean()
        s = users[users['credit_archetype'] == arch][col].std()
        ax.text(i, ax.get_ylim()[1] * 0.97, f'μ={m:.1f}\nσ={s:.1f}',
                ha='center', va='top', fontsize=8, color='black')
plt.suptitle('Activity Feature Distributions by Archetype', fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 3: Categorical Feature Statistics

### 3.1 User-Level Categorical Variables

In [ ]:
# credit_archetype frequency table
print('--- credit_archetype ---')
print(f'Unique values: {labels["credit_archetype"].nunique()}\n')
arch_vc = labels['credit_archetype'].value_counts().reindex(ARCHETYPE_ORDER)
arch_table = pd.DataFrame({
    'Count': arch_vc,
    'Percentage (%)': (arch_vc / len(labels) * 100).round(2)
})
display(arch_table)

# credit_risk_label frequency table
print('\n--- credit_risk_label ---')
print(f'Unique values: {labels["credit_risk_label"].nunique()}\n')
risk_vc = labels['credit_risk_label'].value_counts().sort_index()
risk_table = pd.DataFrame({
    'Label Name': [RISK_LABELS[k] for k in risk_vc.index],
    'Count': risk_vc.values,
    'Percentage (%)': (risk_vc.values / len(labels) * 100).round(2)
}, index=risk_vc.index)
display(risk_table)

# Binary target among borrowers
borrowers = labels[labels['credit_risk_label'] != -1]
defaulters = (borrowers['credit_risk_label'] == 2).sum()
print(f'\nBorrowers only (N={len(borrowers):,}): {defaulters:,} defaulters ({defaulters/len(borrowers)*100:.1f}%)')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = [ARCHETYPE_COLORS[a] for a in ARCHETYPE_ORDER]

# Archetype bar chart
bars = ax1.bar(range(len(arch_vc)), arch_vc.values, color=colors, edgecolor='white', width=0.6)
ax1.set_xticks(range(len(arch_vc)))
ax1.set_xticklabels([ARCHETYPE_LABELS[a] for a in ARCHETYPE_ORDER], rotation=15, ha='right')
for bar, v in zip(bars, arch_vc.values):
    pct = v / len(labels) * 100
    ax1.text(bar.get_x() + bar.get_width()/2, v + 50, f'{v:,}\n({pct:.1f}%)', ha='center', fontsize=9)
ax1.set_ylabel('Count')
ax1.set_title(f'Credit Archetype Distribution (N={len(labels):,})')

# Risk label bar chart
risk_colors = [RISK_COLORS[k] for k in risk_vc.index]
bars2 = ax2.bar([RISK_LABELS[k] for k in risk_vc.index], risk_vc.values,
                color=risk_colors, edgecolor='white', width=0.6)
for bar, v in zip(bars2, risk_vc.values):
    pct = v / len(labels) * 100
    ax2.text(bar.get_x() + bar.get_width()/2, v + 50, f'{v:,}\n({pct:.1f}%)', ha='center', fontsize=9)
ax2.set_ylabel('Count')
ax2.set_title('Credit Risk Label Distribution')

plt.tight_layout()
plt.show()

### 3.2 Transaction-Level Categorical Variables

*(Based on the stratified sample of 500 users.)*

In [ ]:
# Transaction type frequency table
print('--- TRANS. TYPE ---')
print(f'Unique values: {txns["TRANS. TYPE"].nunique()}\n')
type_vc = txns['TRANS. TYPE'].value_counts()
type_table = pd.DataFrame({
    'Count': type_vc,
    'Percentage (%)': (type_vc / len(txns) * 100).round(2)
})
display(type_table)

# Loan provider frequency table (CREDIT rows only)
credit_txns = txns[txns['TRANS. TYPE'] == 'CREDIT']
if 'LOAN_PROVIDER' in credit_txns.columns and len(credit_txns) > 0:
    print(f'\n--- LOAN_PROVIDER (loan disbursements only, N={len(credit_txns):,}) ---')
    print(f'Unique values: {credit_txns["LOAN_PROVIDER"].nunique()}\n')
    prov_vc = credit_txns['LOAN_PROVIDER'].value_counts()
    prov_table = pd.DataFrame({
        'Count': prov_vc,
        'Percentage (%)': (prov_vc / len(credit_txns) * 100).round(2)
    })
    display(prov_table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transaction type bar chart
txn_type_order = ['TRANSFER', 'DEBIT', 'PAYMENT', 'PAYMENT_SEND',
                  'CASH_OUT', 'CASH_IN', 'ADJUSTMENT', 'CREDIT', 'LOAN_REPAYMENT']
present_types = [t for t in txn_type_order if t in type_vc.index]
type_vc_ordered = type_vc.reindex(present_types).dropna()
bars = axes[0].bar(range(len(type_vc_ordered)), type_vc_ordered.values,
                   color=sns.color_palette('Set2', len(type_vc_ordered)), edgecolor='white')
axes[0].set_xticks(range(len(type_vc_ordered)))
axes[0].set_xticklabels(type_vc_ordered.index, rotation=40, ha='right', fontsize=9)
for bar, v in zip(bars, type_vc_ordered.values):
    pct = v / len(txns) * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 10, f'{pct:.1f}%', ha='center', fontsize=8)
axes[0].set_ylabel('Count')
axes[0].set_title('Transaction Type Counts (Sampled)')

# Transaction type mix by archetype (stacked proportions)
type_props = txns.groupby(['credit_archetype', 'TRANS. TYPE']).size().unstack(fill_value=0)
type_props = type_props.div(type_props.sum(axis=1), axis=0)
type_props = type_props.reindex(ARCHETYPE_ORDER)
type_props.plot(kind='bar', stacked=True, ax=axes[1], colormap='Set2', edgecolor='white', linewidth=0.5)
axes[1].set_xticklabels([ARCHETYPE_LABELS[a] for a in ARCHETYPE_ORDER], rotation=15, ha='right')
axes[1].set_ylabel('Proportion')
axes[1].set_title('Transaction Type Mix by Archetype')
axes[1].legend(title='Type', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

### 3.3 Temporal Categorical Variables

In [ ]:
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

# Day-of-week frequency table
print('--- Day of Week ---')
dow_vc = txns['day_of_week'].value_counts().reindex(dow_order)
dow_table = pd.DataFrame({
    'Count': dow_vc,
    'Percentage (%)': (dow_vc / len(txns) * 100).round(2)
})
display(dow_table)

# Hour bucket frequency table
print('\n--- Hour of Day (0-23) ---')
hour_vc = txns['hour'].value_counts().sort_index()
hour_table = pd.DataFrame({
    'Count': hour_vc,
    'Percentage (%)': (hour_vc / len(txns) * 100).round(2)
})
display(hour_table)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Day-of-week bar chart
bars = ax1.bar(range(7), dow_vc.values, color='#3498db', edgecolor='white')
ax1.set_xticks(range(7))
ax1.set_xticklabels([d[:3] for d in dow_order], fontsize=10)
for bar, v in zip(bars, dow_vc.values):
    pct = v / len(txns) * 100
    ax1.text(bar.get_x() + bar.get_width()/2, v + 20, f'{pct:.1f}%', ha='center', fontsize=8)
ax1.set_ylabel('Transaction Count')
ax1.set_title('Transaction Count by Day of Week')

# Hour-of-day bar chart
hour_colors = ['#2c3e50' if 6 <= h < 22 else '#95a5a6' for h in hour_vc.index]
ax2.bar(hour_vc.index, hour_vc.values, color=hour_colors, edgecolor='white', width=0.8)
ax2.set_xlabel('Hour of Day')
ax2.set_ylabel('Transaction Count')
ax2.set_title('Transaction Count by Hour of Day')
ax2.set_xticks(range(0, 24, 2))
legend_elements = [
    mpatches.Patch(facecolor='#2c3e50', label='Daytime (06:00-21:59)'),
    mpatches.Patch(facecolor='#95a5a6', label='Night (22:00-05:59)'),
]
ax2.legend(handles=legend_elements, fontsize=9)

plt.tight_layout()
plt.show()

---
## Section 4: Key Numerical Distributions

Distribution plots for transaction amounts and loan-specific data.

In [ ]:
amounts = txns['AMOUNT'][txns['AMOUNT'] > 0.5]

print('--- AMOUNT (all transaction types) ---')
print(amounts.describe().round(2).to_string())
print(f'CV:      {amounts.std() / amounts.mean():.4f}')
print(f'Skew:    {amounts.skew():.4f}')
print(f'Kurt:    {amounts.kurt():.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log-scale histogram
ax1.hist(np.log10(amounts), bins=60, density=True, alpha=0.7, color='#3498db', edgecolor='white')
ax1.set_xlabel('log10(Amount in GHS)')
ax1.set_ylabel('Density')
ax1.set_title('Transaction Amount Distribution (Log Scale)')
ax1.axvline(np.log10(amounts.median()), color='#e74c3c', linestyle='--', linewidth=2,
            label=f'Median: GHS {amounts.median():.1f}')
ax1.axvline(np.log10(amounts.mean()), color='#2ecc71', linestyle='--', linewidth=2,
            label=f'Mean: GHS {amounts.mean():.1f}')
ax1.legend(fontsize=10)

# Box plots by transaction type
txn_type_order_plot = ['TRANSFER', 'DEBIT', 'PAYMENT', 'PAYMENT_SEND',
                        'CASH_OUT', 'CASH_IN', 'CREDIT', 'LOAN_REPAYMENT']
present = [t for t in txn_type_order_plot if t in txns['TRANS. TYPE'].unique()]
sns.boxplot(data=txns[txns['AMOUNT'] > 0.5], x='TRANS. TYPE', y='AMOUNT',
            order=present, palette='Set2', ax=ax2, showfliers=False)
ax2.set_xticklabels(present, rotation=40, ha='right')
ax2.set_ylabel('Amount (GHS)')
ax2.set_title('Amount Distribution by Transaction Type (IQR)')

plt.tight_layout()
plt.show()

In [ ]:
credit_txns = txns[txns['TRANS. TYPE'] == 'CREDIT']
repayment_txns = txns[txns['TRANS. TYPE'] == 'LOAN_REPAYMENT']

print('--- Loan Disbursement Amounts (CREDIT transactions) ---')
if len(credit_txns) > 0:
    print(credit_txns['AMOUNT'].describe().round(2).to_string())
    print(f'CV: {credit_txns["AMOUNT"].std() / credit_txns["AMOUNT"].mean():.4f}')

print('\n--- Loan Repayment Amounts (LOAN_REPAYMENT transactions) ---')
if len(repayment_txns) > 0:
    print(repayment_txns['AMOUNT'].describe().round(2).to_string())
    print(f'CV: {repayment_txns["AMOUNT"].std() / repayment_txns["AMOUNT"].mean():.4f}')

print('\n--- Loans Taken per User (borrowers only) ---')
borrower_users = users[users['credit_risk_label'] != -1]
print(borrower_users['loans_taken'].describe().round(2).to_string())

---
## Section 5: Feature Correlations

Pearson correlation matrix for a focused set of interpretable features plus the binary default target. Red = positive correlation, Blue = negative.

In [ ]:
borrower_data = users[users['credit_risk_label'] != -1].copy()
borrower_data['default'] = (borrower_data['credit_risk_label'] == 2).astype(int)

focus_features = [
    'total_volume', 'avg_transaction_amount', 'cv_transaction_amount',
    'pct_transfers', 'pct_debits', 'pct_cashouts',
    'avg_hours_between_txns', 'pct_weekend_txns', 'pct_night_txns',
    'avg_balance', 'balance_volatility', 'pct_low_balance_txns',
    'unique_recipients', 'recipient_concentration',
    'transactions_per_day', 'default'
]

corr_data = borrower_data[focus_features].corr()

fig, ax = plt.subplots(figsize=(13, 11))
mask = np.triu(np.ones_like(corr_data, dtype=bool), k=1)
sns.heatmap(
    corr_data, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, ax=ax, vmin=-1, vmax=1,
    xticklabels=[f.replace('_', ' ') for f in focus_features],
    yticklabels=[f.replace('_', ' ') for f in focus_features],
    square=True, linewidths=0.5
)
ax.set_title('Feature Correlation Matrix (Borrowers Only)', fontsize=13)
plt.tight_layout()
plt.show()

# Top correlations with default
corr_with_default = corr_data['default'].drop('default').sort_values(key=abs, ascending=False)
print('Top 10 features by |correlation| with default:')
print(corr_with_default.head(10).round(4).to_string())

---
## Section 6: Per-Archetype Summary Statistics

Mean and standard deviation for key features, grouped by credit archetype.

In [ ]:
summary_cols = [
    'total_transactions', 'total_volume', 'avg_transaction_amount',
    'avg_balance', 'balance_volatility', 'pct_low_balance_txns',
    'unique_recipients', 'transactions_per_day'
]

agg_mean = users.groupby('credit_archetype')[summary_cols].mean().reindex(ARCHETYPE_ORDER)
agg_std  = users.groupby('credit_archetype')[summary_cols].std().reindex(ARCHETYPE_ORDER)

# Combined mean ± std table
combined = pd.DataFrame(index=[ARCHETYPE_LABELS[a] for a in ARCHETYPE_ORDER],
                         columns=[c.replace('_', ' ').title() for c in summary_cols])
for col, label in zip(summary_cols, combined.columns):
    for arch in ARCHETYPE_ORDER:
        m = agg_mean.loc[arch, col]
        s = agg_std.loc[arch, col]
        combined.loc[ARCHETYPE_LABELS[arch], label] = f'{m:.2f} ± {s:.2f}'

print('Mean ± Std Dev by Archetype')
display(combined)

In [ ]:
# Styled mean-only table with gradient for easy scanning
mean_table = agg_mean.copy()
mean_table.index = [ARCHETYPE_LABELS[a] for a in ARCHETYPE_ORDER]
mean_table.columns = [c.replace('_', ' ').title() for c in summary_cols]

try:
    display(mean_table.round(2).style.background_gradient(cmap='YlOrRd', axis=0).format('{:.2f}'))
except Exception:
    display(mean_table.round(2))

In [ ]:
print(f'Descriptive statistics complete.')
print(f'Dataset: {len(users):,} users | {len(num_cols)} numeric features | {n_txn_files:,} transaction files')
print(f'Sampled transactions: {len(txns):,} rows from {len(set(sample_ids))} users')